In [1]:
# ── Cell 1: Mount Drive & clone repo ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from getpass import getpass
import subprocess, os

token    = getpass("GitHub token: ")
repo_dir = '/content/notebooks_meta'

if not os.path.exists(repo_dir):
    subprocess.run(
        ['git', 'clone',
         f'https://{token}@github.com/jonyxdxp/notebooks_meta.git',
         repo_dir], check=True)

%cd {repo_dir}
!git pull origin main

Mounted at /content/drive
GitHub token: ··········
/content/notebooks_meta
From https://github.com/jonyxdxp/notebooks_meta
 * branch            main       -> FETCH_HEAD
Already up to date.


In [2]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 817.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:

# ── Cell 3: Imports ───────────────────────────────────────────────────────────

import copy
import sys
import typing

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import datasets
import transformers
import tokenizers

sys.path.insert(0, '/content/notebooks_meta/v4/1')

from cog_arch.encoder import Encoder
from losses import VICRegLoss, BCS   # BCS kept as optional alternative


In [4]:
# ── Cell 6: Device ────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cpu


In [5]:
# ── Cell 5: Config (single place to change hyperparams) ───────────────────────

# Helper class to allow dot notation access to dictionary keys
class _C:
    def __init__(self, d):
        for k, v in d.items():
            setattr(self, k, _C(v) if isinstance(v, dict) else v)

CFG_dict = dict(
    # Model
    hidden_size  = 256,
    num_heads    = 4,       # 256 / 4 = 64 per head
    num_layers   = 4,
    max_seq_len  = 128,
    mlp_ratio    = 4.0,

    # JEPA masking
    num_target_spans   = 4,
    target_span_length = 8,

    # Training
    lr           = 1e-4,
    weight_decay = 0.05,
    n_epochs     = 20,
    ema_decay     = 0.996,
    batch_size   = 64,
    eval_batch   = 128,
    num_workers  = 0,

    # VICReg
    std_coeff = 50.0,   # was 25.0
    cov_coeff = 1.0,

    # Paths
    cache_dir    = '/content/drive/MyDrive/data/cache',
    ckpt_dir     = '/content/notebooks_meta/v4/1/checkpoints',
    raw_data_dir = '/content/data/dailydialog_processed', # Updated to new, consistent path for extracted data
    tokenizer_name = 'bert-base-uncased',
)

CFG = _C(CFG_dict) # Create CFG object supporting dot notation

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # Define DEVICE as a global variable

os.makedirs(CFG.ckpt_dir, exist_ok=True) # Access via dot notation

In [6]:
tokenizer = transformers.AutoTokenizer.from_pretrained(CFG.tokenizer_name)
# BERT already has [MASK], [PAD], [CLS], [SEP] — nothing to patch.
# For GPT-style tokenizers that lack these tokens, add them here.

VOCAB_SIZE = tokenizer.vocab_size
print(f"Vocab : {VOCAB_SIZE}  |  mask_token_id : {tokenizer.mask_token_id}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab : 30522  |  mask_token_id : 103


In [7]:
# ── Cell 7: Dataset ───────────────────────────────────────────────────────────

def get_dailydialog_dataset(
    cache_dir: str,
    tokenizer,
    block_size: int = 128,
    num_proc: int = 1,
    raw_data_dir: str = '/content/notebooks_meta/v4/1/data/raw/ijcnlp_dailydialog',
) -> datasets.DatasetDict:

    _cache_path = os.path.join(cache_dir, f'dailydialog_jepa_bs{block_size}')
    if os.path.exists(_cache_path):
        print(f'Loading from cache: {_cache_path}')
        return datasets.load_from_disk(_cache_path).with_format('torch')

    split_txts = {
        'train':      os.path.join(raw_data_dir, 'train',      'dialogues_train.txt'),
        'validation': os.path.join(raw_data_dir, 'validation', 'dialogues_validation.txt'),
        'test':       os.path.join(raw_data_dir, 'test',       'dialogues_test.txt'),
    }

    # ── Step 1: download raw data if txt files not already on disk ────────────
    if not all(os.path.exists(p) for p in split_txts.values()):
        print('Downloading DailyDialog …')

        raw = None

        for repo in ['benjaminbeilharz/better_daily_dialog']:
            try:
                print(f'  trying {repo} …')
                raw = datasets.load_dataset(repo)

                import pandas as pd
                split_dicts = {}
                for split in raw:
                    df = raw[split].to_pandas()
                    dialogs = (
                        df.sort_values(['dialog_id', 'turn_type'])
                          .groupby('dialog_id')['utterance']
                          .apply(list)
                          .tolist()
                    )
                    split_dicts[split] = datasets.Dataset.from_dict({'dialog': dialogs})
                raw = datasets.DatasetDict(split_dicts)

                print(f'  ✓ loaded {sum(len(raw[s]) for s in raw):,} dialogues from {repo}')
                break
            except Exception as e:
                print(f'  ✗ {e}')
                raw = None

        if raw is None:
            raise RuntimeError("All dataset sources failed. Check your internet connection.")

        for split, txt_path in split_txts.items():
            os.makedirs(os.path.dirname(txt_path), exist_ok=True)
            with open(txt_path, 'w', encoding='utf-8') as f:
                for example in raw[split]:
                    turns = [t.strip() for t in example['dialog'] if t.strip()]
                    if turns:
                        f.write(' __eou__ '.join(turns) + ' __eou__\n')
            with open(txt_path) as f:
                n = sum(1 for l in f if l.strip())
            print(f'  wrote {split} ({n:,} dialogues) → {txt_path}')

    # ── Step 2: read utterances ───────────────────────────────────────────────
    def _load_utterances(filepath):
        utterances = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                turns = [t.strip() for t in line.split('__eou__') if t.strip()]
                utterances.extend(turns)
        return utterances

    def _tokenize(examples):
        return tokenizer(
            examples['text'],
            max_length=block_size,
            padding='max_length',
            truncation=True,
            add_special_tokens=True,
            return_attention_mask=True,
            return_token_type_ids=False,
        )

    print('Building DailyDialog JEPA dataset …')
    tokenized_splits = {}
    for split, txt_path in split_txts.items():
        utterances = _load_utterances(txt_path)
        print(f'  {split:12s}: {len(utterances):>6,} utterances')
        raw_ds = datasets.Dataset.from_dict({'text': utterances})
        tokenized_splits[split] = raw_ds.map(
            _tokenize, batched=True, num_proc=num_proc,
            remove_columns=['text'], desc=f'Tokenizing {split}')

    dataset_dict = datasets.DatasetDict(tokenized_splits)
    os.makedirs(cache_dir, exist_ok=True)
    dataset_dict.save_to_disk(_cache_path)
    print(f'Saved to cache: {_cache_path}')
    return dataset_dict.with_format('torch')

In [8]:
# ── Cell 8: Collator ──────────────────────────────────────────────────────────

class JEPAMaskCollator:
    """
    Returns context (masked) and target (clean) views of the same sequence.

    Keys returned:
      context_input_ids      (B, L) — masked tokens replaced with [MASK]
      context_attention_mask (B, L)
      target_input_ids       (B, L) — original, unmasked
      target_attention_mask  (B, L)
      target_mask            (B, L) bool — True at positions that were masked
    """

    def __init__(
        self,
        mask_token_id: int,
        pad_token_id: int,
        num_target_spans: int = 4,
        target_span_length: int = 8,
    ):
        self.mask_token_id    = mask_token_id
        self.pad_token_id     = pad_token_id
        self.num_target_spans = num_target_spans
        self.target_span_len  = target_span_length

    def __call__(self, batch):
        input_ids      = torch.stack([b['input_ids']      for b in batch])
        attention_mask = torch.stack([b['attention_mask'] for b in batch])
        B, L = input_ids.shape

        context_input_ids = input_ids.clone()
        target_mask       = torch.zeros(B, L, dtype=torch.bool)

        for i in range(B):
            valid_len = int(attention_mask[i].sum().item())
            for s, e in self._sample_spans(1, valid_len - 1):
                target_mask[i, s:e] = True
            context_input_ids[i, target_mask[i]] = self.mask_token_id

        return {
            'context_input_ids':      context_input_ids,
            'context_attention_mask': attention_mask,
            'target_input_ids':       input_ids,
            'target_attention_mask':  attention_mask,
            'target_mask':            target_mask,
        }

    def _sample_spans(self, maskable_start, maskable_end):
        region_len = maskable_end - maskable_start
        if region_len <= 0:
            return []
        span_len  = min(self.target_span_len, region_len)
        available = list(range(maskable_start, maskable_end - span_len + 1))
        spans = []
        for _ in range(self.num_target_spans):
            if not available:
                break
            idx = torch.randint(len(available), (1,)).item()
            s, e = available[idx], available[idx] + span_len
            spans.append((s, e))
            available = [x for x in available if (x + span_len <= s) or (x >= e)]
        return spans

In [9]:
# dataloader


# ---------------------------------------------------------------------------
# JEPA dataloaders
# ---------------------------------------------------------------------------

def get_jepa_dataloaders(
    cfg_obj, # Changed parameter name to avoid conflict with global CFG
    tokenizer,
    skip_train: bool = False,
    skip_valid: bool = False,
    valid_seed: typing.Optional[int] = None,
):
  """Build train / validation DataLoaders for JEPA phase-1 training.

  Expects cfg_obj.num_target_spans and cfg_obj.target_span_length
  (defaults: 4 and 8).  Batch items have keys produced by JEPAMaskCollator.
  """
  num_gpus  = max(torch.cuda.device_count(), 1)
  block_size = cfg_obj.max_seq_len # Access via dot notation

  if cfg_obj.eval_batch % num_gpus != 0:
    raise ValueError(
      f'Eval batch size {cfg_obj.eval_batch} '
      f'not divisible by {num_gpus} GPUs.')

  dataset_dict = get_dailydialog_dataset(
    cache_dir=cfg_obj.cache_dir,
    tokenizer=tokenizer,
    block_size=block_size,
    raw_data_dir=cfg_obj.raw_data_dir, # Pass raw_data_dir explicitly
  )

  if tokenizer.mask_token_id is None:
    raise ValueError(
      f'Tokenizer must have a mask_token for JEPA masking: {tokenizer}')

  collator = JEPAMaskCollator(
    mask_token_id=tokenizer.mask_token_id,
    pad_token_id=tokenizer.pad_token_id,
    num_target_spans=cfg_obj.num_target_spans,
    target_span_length=cfg_obj.target_span_length,
  )

  train_loader = valid_loader = None

  if not skip_train:
    train_loader = torch.utils.data.DataLoader(
    dataset_dict['train'],
    batch_size=cfg_obj.batch_size,
    num_workers=cfg_obj.num_workers,
    pin_memory=True,
    shuffle=True,
    persistent_workers=cfg_obj.num_workers > 0,
    collate_fn=collator,
    )
    train_loader.tokenizer = tokenizer

  if not skip_valid:
    generator    = torch.Generator().manual_seed(valid_seed) if valid_seed else None
    valid_loader = torch.utils.data.DataLoader(
    dataset_dict['validation'],
    batch_size=cfg_obj.eval_batch,
    num_workers=cfg_obj.num_workers,
    pin_memory=True,
    shuffle=valid_seed is not None,
    generator=generator,
    persistent_workers=cfg_obj.num_workers > 0,
    collate_fn=collator,
    )
    valid_loader.tokenizer = tokenizer

  return train_loader, valid_loader

In [10]:
# ── Cell 9: Build dataloaders ─────────────────────────────────────────────────

train_loader, val_loader = get_jepa_dataloaders(
    cfg_obj    = CFG,
    tokenizer  = tokenizer,
)

print(f"Train batches : {len(train_loader)}  |  Val batches : {len(val_loader)}")

Loading from cache: /content/drive/MyDrive/data/cache/dailydialog_jepa_bs128
Train batches : 1363  |  Val batches : 64


In [11]:
# ── Cell 10: Models ───────────────────────────────────────────────────────────

context_encoder = Encoder(
    vocab_size   = VOCAB_SIZE,
    hidden_size  = CFG.hidden_size,
    num_heads    = CFG.num_heads,
    num_layers   = CFG.num_layers,
    max_seq_len  = CFG.max_seq_len,
).to(DEVICE)

target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for p in target_encoder.parameters():
    p.requires_grad = False   # updated only via EMA

print(f"Params (context encoder) : {sum(p.numel() for p in context_encoder.parameters()):,}")

Params (context encoder) : 12,049,408


In [12]:
# ── Cell 11: Loss / optimizer / scheduler ────────────────────────────────────

# loss_fn = VICRegLoss(std_coeff=CFG.std_coeff, cov_coeff=CFG.cov_coeff)

loss_fn = BCS(lmbd=10.0)   # lmbd controls Gaussianity regularization vs invariance



optimizer = AdamW(
    context_encoder.parameters(),
    lr           = CFG.lr,
    weight_decay = CFG.weight_decay,
)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max  = CFG.n_epochs * 2,   # was n_epochs — decays over 2x the actual run
    eta_min = CFG.lr * 0.3,      # was 0.1 — don't let it drop as low
)

In [13]:
# ── Cell 12: Helpers ──────────────────────────────────────────────────────────

def ema_update(ctx_enc, tgt_enc, decay=CFG.ema_decay):
    """Exponential moving average: tgt ← decay*tgt + (1-decay)*ctx"""
    with torch.no_grad():
        for p_c, p_t in zip(ctx_enc.parameters(), tgt_enc.parameters()):
            p_t.data.mul_(decay).add_(p_c.data, alpha=1.0 - decay)


def masked_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    Mean-pool hidden states at masked positions per batch item.

    Args:
        hidden : (B, L, D)  encoder output
        mask   : (B, L)     bool — True at target positions

    Returns:
        pooled : (B, D)
    """
    mask_f = mask.unsqueeze(-1).float()            # (B, L, 1)
    summed = (hidden * mask_f).sum(dim=1)          # (B, D)
    count  = mask_f.sum(dim=1).clamp(min=1)        # (B, 1)
    return summed / count


def unpack(batch):
    return (
        batch['context_input_ids'].to(DEVICE),
        batch['context_attention_mask'].to(DEVICE),
        batch['target_input_ids'].to(DEVICE),
        batch['target_attention_mask'].to(DEVICE),
        batch['target_mask'].to(DEVICE),
    )

In [14]:
# ── Cell 13: Train / eval steps ───────────────────────────────────────────────

def forward_step(batch):
    """
    Single JEPA forward pass (no predictor).

    Returns a dict of scalar losses.
    """
    ctx_ids, ctx_mask, tgt_ids, tgt_mask, span_mask = unpack(batch)

    # ── context encoder (grad flows here) ────────────────────────────────────
    # Encoder is expected to return (sequence_hidden, pooled) or just hidden.
    # Adjust the indexing below to match your Encoder's actual return signature.
    ctx_hidden = context_encoder(ctx_ids, attention_mask=ctx_mask)   # (B, L, D)
    if isinstance(ctx_hidden, tuple):
        ctx_hidden = ctx_hidden[0]

    # ── target encoder (no grad, EMA) ────────────────────────────────────────
    with torch.no_grad():
        tgt_hidden = target_encoder(tgt_ids, attention_mask=tgt_mask)
        if isinstance(tgt_hidden, tuple):
            tgt_hidden = tgt_hidden[0]

    # ── pool only at masked positions ────────────────────────────────────────
    z_ctx = masked_pool(ctx_hidden, span_mask)   # (B, D)
    z_tgt = masked_pool(tgt_hidden, span_mask)   # (B, D)

    # ── loss ─────────────────────────────────────────────────────────────────
    return loss_fn(z_ctx, z_tgt)   # dict with 'loss', 'std_loss', 'cov_loss', etc.


@torch.no_grad()
def eval_epoch(loader):
    context_encoder.eval()
    totals = {}
    n = 0
    for batch in loader:
        loss_dict = forward_step(batch)
        for k, v in loss_dict.items():
            totals[k] = totals.get(k, 0.0) + v.item()
        n += 1
    return {k: v / n for k, v in totals.items()}


def train_epoch(loader, epoch):
    context_encoder.train()
    totals = {}
    n = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch:02d}', leave=False)
    for batch in pbar:
        loss_dict = forward_step(batch)
        loss = loss_dict['loss']

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(context_encoder.parameters(), 1.0)
        optimizer.step()

        # EMA update after every gradient step
        ema_update(context_encoder, target_encoder)

        for k, v in loss_dict.items():
            totals[k] = totals.get(k, 0.0) + v.item()
        n += 1

        pbar.set_postfix({k: f'{v.item():.4f}' for k, v in loss_dict.items()})

    return {k: v / n for k, v in totals.items()}

# ── Cell 14: Checkpointing ────────────────────────────────────────────────────

def save_checkpoint(epoch, metrics):
    path = os.path.join(CFG.ckpt_dir, f'epoch_{epoch:03d}.pt')
    torch.save({
        'epoch':           epoch,
        'context_encoder': context_encoder.state_dict(),
        'target_encoder':  target_encoder.state_dict(),
        'optimizer':       optimizer.state_dict(),
        'scheduler':       scheduler.state_dict(),
        'metrics':         metrics,
        'cfg':             CFG,
    }, path)
    print(f'  ✓ saved → {path}')


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)
    context_encoder.load_state_dict(ckpt['context_encoder'])
    target_encoder.load_state_dict(ckpt['target_encoder'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    print(f'  ✓ resumed from epoch {ckpt["epoch"]}')
    return ckpt['epoch']

In [15]:
# ── Cell 15: Training loop ────────────────────────────────────────────────────

history = {
    'train_loss': [], 'train_bcs': [], 'train_inv': [],
    'val_loss':   [], 'val_bcs':   [], 'val_inv':   [],
    'lr':         [],
}

print(f'\n{"="*60}')
print(f'  Text JEPA — {CFG.n_epochs} epochs   device={DEVICE}')
print(f'{"="*60}\n')

# Resume from checkpoint if one exists
import glob
start_epoch = 1
ckpts = sorted(glob.glob(os.path.join(CFG.ckpt_dir, 'epoch_*.pt')))
if ckpts:
    latest = ckpts[-1]
    print(f'Resuming from {latest} …')
    start_epoch = load_checkpoint(latest) + 1
    print(f'  starting at epoch {start_epoch}')
else:
    print('No checkpoint found, starting from scratch.')

best_val_loss = float('inf')

for epoch in range(start_epoch, CFG.n_epochs + 1):
    train_metrics = train_epoch(train_loader, epoch)
    val_metrics   = eval_epoch(val_loader)
    scheduler.step()

    history['train_loss'].append(train_metrics.get('loss', 0.0))
    history['train_bcs'].append(train_metrics.get('bcs_loss', 0.0))
    history['train_inv'].append(train_metrics.get('invariance_loss', 0.0))
    history['val_loss'].append(val_metrics.get('loss', 0.0))
    history['val_bcs'].append(val_metrics.get('bcs_loss', 0.0))
    history['val_inv'].append(val_metrics.get('invariance_loss', 0.0))
    history['lr'].append(optimizer.param_groups[0]['lr'])

    print(
        f'Epoch {epoch:02d}/{CFG.n_epochs}  '
        f'train_loss={train_metrics["loss"]:.4f}  '
        f'val_loss={val_metrics["loss"]:.4f}  '
        f'bcs={train_metrics.get("bcs_loss", 0):.4f}  '
        f'inv={train_metrics.get("invariance_loss", 0):.4f}  '
        f'lr={optimizer.param_groups[0]["lr"]:.2e}'
    )

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        save_checkpoint('best', val_metrics)
        print(f'  ★ new best val_loss={best_val_loss:.4f}')

    if epoch % 5 == 0:
        save_checkpoint(epoch, {**train_metrics, **{f'val_{k}': v for k, v in val_metrics.items()}})

print('\nTraining complete.')
save_checkpoint(CFG.n_epochs, {})


  Text JEPA — 20 epochs   device=cpu

No checkpoint found, starting from scratch.


KeyboardInterrupt: 

In [ ]:
def plot_training(history, save_path=None):
    epochs = range(1, len(history['train_loss']) + 1)
    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    def _ax(row, col, title, ylabel='Loss'):
        ax = fig.add_subplot(gs[row, col])
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
        return ax

    # 1. Total loss
    ax = _ax(0, 0, 'Total Loss')
    ax.plot(epochs, history['train_loss'], 'b-o', label='train', ms=4)
    ax.plot(epochs, history['val_loss'],   'r-o', label='val',   ms=4)
    ax.legend()

    # 2. BCS (Gaussianity) loss
    ax = _ax(0, 1, 'BCS Loss  (↓ = more Gaussian, no collapse)')
    ax.plot(epochs, history['train_bcs'], 'b-o', label='train', ms=4)
    ax.plot(epochs, history['val_bcs'],   'r-o', label='val',   ms=4)
    ax.axhline(0.0, color='k', ls='--', alpha=0.4)
    ax.legend()

    # 3. Invariance loss
    ax = _ax(0, 2, 'Invariance Loss  (↓ = ctx/tgt closer)')
    ax.plot(epochs, history['train_inv'], 'b-o', label='train', ms=4)
    ax.plot(epochs, history['val_inv'],   'r-o', label='val',   ms=4)
    ax.legend()

    # 4. Train/val gap
    ax = _ax(1, 0, 'Val − Train Gap  (red=overfit, green=underfit)')
    gap    = [v - t for t, v in zip(history['train_loss'], history['val_loss'])]
    colors = ['red' if g > 0 else 'green' for g in gap]
    ax.bar(epochs, gap, color=colors, alpha=0.7)
    ax.axhline(0, color='k', ls='--', alpha=0.5)

    # 5. Learning rate
    ax = _ax(1, 1, 'Learning Rate Schedule', ylabel='LR')
    ax.plot(epochs, history['lr'], color='orange', lw=1.5)
    ax.set_yscale('log')

    # 6. Config summary
    ax = fig.add_subplot(gs[1, 2])
    ax.axis('off')
    ax.text(0.5, 0.5,
            f'Loss     : BCS  (lmbd=10.0)\n'
            f'EMA decay: {CFG.ema_decay}\n'
            f'Spans×len: {CFG.num_target_spans}×{CFG.target_span_length}\n'
            f'Hidden   : {CFG.hidden_size}  Layers: {CFG.num_layers}\n'
            f'LR       : {CFG.lr}  Epochs: {CFG.n_epochs}',
            ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))

    plt.suptitle(
        f'Text JEPA Stage-1 — {len(epochs)} epochs', fontsize=13, fontweight='bold')

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f'Plot saved → {save_path}')

    plt.show()
    plt.close()


plot_training(history, save_path='/content/drive/MyDrive/text_jepa_training.png')

In [ ]:
!ls /content/notebooks_meta/v3/first_phase

In [ ]:
# ── Cell: Masked Span Recovery Evaluation ────────────────────────────────────
#
# For each utterance in the test set:
#   1. Create a masked version (same collator as training)
#   2. Encode masked → z_ctx  (context encoder, trained)
#   3. Encode original → z_tgt (context encoder, trained)
#   4. Measure cosine similarity sim(z_ctx, z_tgt)
#   5. Compare against sim(z_ctx, z_random) for N random distractors
#
# Metric: Recall@K — is the correct original in the top-K most similar?
# If the encoder learned to predict masked spans, sim(ctx, correct) >> sim(ctx, random)

import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

# ── Encode helper (mean pool, no grad) ───────────────────────────────────────

@torch.no_grad()
def encode_batch(encoder, input_ids, attention_mask):
    hidden = encoder(input_ids, attention_mask=attention_mask)
    if isinstance(hidden, tuple):
        hidden = hidden[0]                              # (B, L, D)
    mask_f = attention_mask.unsqueeze(-1).float()
    return F.normalize(
        (hidden * mask_f).sum(1) / mask_f.sum(1).clamp(min=1),
        dim=-1
    )                                                   # (B, D) L2-normalized

# ── Build a pool of test embeddings (originals) ───────────────────────────────
# We'll use these as the distractor bank

print('Building test embedding pool …')

# Use the plain (non-JEPA) dataloader — we want clean input_ids + attention_mask
plain_test_loader = torch.utils.data.DataLoader(
    dataset['test'],
    batch_size  = 256,
    shuffle     = False,
    num_workers = 0,
    collate_fn  = lambda batch: {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
    }
)

all_embeddings_trained = []
all_embeddings_random  = []

for batch in tqdm(plain_test_loader, desc='Encoding test pool'):
    ids  = batch['input_ids'].to(DEVICE)
    mask = batch['attention_mask'].to(DEVICE)
    all_embeddings_trained.append(encode_batch(context_encoder, ids, mask).cpu())
    all_embeddings_random.append(encode_batch(random_encoder,   ids, mask).cpu())

pool_trained = torch.cat(all_embeddings_trained, dim=0)  # (N_test, D)
pool_random  = torch.cat(all_embeddings_random,  dim=0)

print(f'  pool size: {pool_trained.shape[0]:,} utterances  dim={pool_trained.shape[1]}')

# ── JEPA collator to produce masked views ─────────────────────────────────────
# Reuse the same collator from training — identical masking distribution

jepa_test_loader = torch.utils.data.DataLoader(
    dataset['test'],
    batch_size  = 256,
    shuffle     = False,
    num_workers = 0,
    collate_fn  = collator,   # JEPAMaskCollator from training
)

# ── Recall@K evaluation ───────────────────────────────────────────────────────

def recall_at_k(ctx_embs, pool_embs, ks=(1, 5, 10)):
    """
    For each query i, rank all pool entries by cosine similarity.
    Recall@K = fraction of queries where correct entry (index i) is in top-K.
    ctx_embs and pool_embs are L2-normalized → dot product = cosine sim.
    """
    N = ctx_embs.shape[0]
    # Full similarity matrix (N, N)
    sim = ctx_embs @ pool_embs.T    # (N, N)
    results = {}
    for k in ks:
        topk_indices = sim.topk(k, dim=1).indices   # (N, K)
        correct = torch.arange(N).unsqueeze(1)       # (N, 1)
        hits = (topk_indices == correct).any(dim=1).float()
        results[f'R@{k}'] = hits.mean().item() * 100
    return results


print('\nEncoding masked (context) views …')

ctx_embs_trained = []
ctx_embs_random  = []

for batch in tqdm(jepa_test_loader, desc='Encoding masked views'):
    ctx_ids  = batch['context_input_ids'].to(DEVICE)
    ctx_mask = batch['context_attention_mask'].to(DEVICE)
    ctx_embs_trained.append(encode_batch(context_encoder, ctx_ids, ctx_mask).cpu())
    ctx_embs_random.append(encode_batch(random_encoder,   ctx_ids, ctx_mask).cpu())

ctx_trained = torch.cat(ctx_embs_trained, dim=0)   # (N_test, D)
ctx_random  = torch.cat(ctx_embs_random,  dim=0)

# ── Run Recall@K ──────────────────────────────────────────────────────────────

print('\nComputing Recall@K …')

r_trained = recall_at_k(ctx_trained, pool_trained)
r_random  = recall_at_k(ctx_random,  pool_random)

# Random chance baseline (uniform retrieval over N items)
N = pool_trained.shape[0]
r_chance = {f'R@{k}': k / N * 100 for k in (1, 5, 10)}

print(f'\n{"="*52}')
print(f'  Masked Span Recovery — Recall@K  (N={N:,})')
print(f'{"="*52}')
print(f'  {"":20s}  R@1      R@5      R@10')
print(f'  {"Random chance":20s}  '
      f'{r_chance["R@1"]:.3f}%  {r_chance["R@5"]:.3f}%  {r_chance["R@10"]:.3f}%')
print(f'  {"Random encoder":20s}  '
      f'{r_random["R@1"]:.2f}%   {r_random["R@5"]:.2f}%   {r_random["R@10"]:.2f}%')
print(f'  {"Trained encoder":20s}  '
      f'{r_trained["R@1"]:.2f}%   {r_trained["R@5"]:.2f}%   {r_trained["R@10"]:.2f}%')
print(f'  {"Gap (trained-random)":20s}  '
      f'{r_trained["R@1"]-r_random["R@1"]:+.2f}%   '
      f'{r_trained["R@5"]-r_random["R@5"]:+.2f}%   '
      f'{r_trained["R@10"]-r_random["R@10"]:+.2f}%')
print(f'{"="*52}')

# ── Also report mean reciprocal rank (MRR) ────────────────────────────────────

@torch.no_grad()
def mean_reciprocal_rank(ctx_embs, pool_embs):
    sim  = ctx_embs @ pool_embs.T          # (N, N)
    N    = sim.shape[0]
    correct = torch.arange(N)
    ranks = (sim.argsort(dim=1, descending=True) == correct.unsqueeze(1)).float().argmax(dim=1) + 1
    return (1.0 / ranks.float()).mean().item()

mrr_trained = mean_reciprocal_rank(ctx_trained, pool_trained)
mrr_random  = mean_reciprocal_rank(ctx_random,  pool_random)

print(f'\n  MRR  random={mrr_random:.4f}  trained={mrr_trained:.4f}  '
      f'gap={mrr_trained - mrr_random:+.4f}')
print(f'  (MRR=1.0 means always ranks correct first; random chance ≈ {1/N:.4f})')